In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import userdata
import os

# Token'ı Colab secrets'a manuel ekleyin:
# Sol tarafta 🔑 (Key) simgesine tıklayın
# "Add new secret" > Name: HF_TOKEN > Value: YOUR_TOKEN_HERE
# hf_REDACTED
# Sonra bu kodu çalıştırın:
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("✅ Hugging Face token yüklendi!")
except:
    print("❌ Token bulunamadı. Lütfen Colab secrets'a ekleyin.")

✅ Hugging Face token yüklendi!


In [3]:
# SAM3 ve bağımlılıklarını kur
!pip install -q torch torchvision torchaudio
!pip install -q pycocotools timm einops psutil decord
!pip install -q pillow matplotlib
print("✅ Kurulum tamamlandı!")
gerekli = ['pycocotools', 'timm', 'einops', 'psutil','decord']

for paket in gerekli:
    try:
        __import__(paket)
        print(f"✅ {paket} yüklü")
    except ImportError:
        print(f"❌ {paket} yüklü değil - yükleniyor...")
        !pip install -q {paket}
        print(f"✅ {paket} kuruldu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 106.3 MB/s eta 0:00:00
✅ Kurulum tamamlandı!
✅ pycocotools yüklü
✅ timm yüklü
✅ einops yüklü
✅ psutil yüklü
✅ decord yüklü


In [4]:
# Önce mevcut sam3 varsa temizle
!rm -rf sam3

# GitHub'dan klonla
!git clone https://github.com/facebookresearch/sam3.git

# Klonlama başarılı mı kontrol et
import os
if os.path.exists('sam3'):
    print("✅ SAM3 klasörü klonlandı")

    # sam3 klasörüne gir ve kur
    %cd sam3
    !pip install -e .

    print("\n=== Kurulum Kontrolü ===")
    # Kontrol et
    try:
        from sam3.model_builder import build_sam3_image_model
        from sam3.model.sam3_image_processor import Sam3Processor
        print("✅ SAM3 modülleri başarıyla import edildi!")
    except Exception as e:
        print(f"❌ Import hatası: {e}")
else:
    print("❌ Klonlama başarısız. Repo adı yanlış olabilir.")
    print("Alternatif olarak SAM2'yi deneyelim:")
    !git clone https://github.com/facebookresearch/sam2.git

Cloning into 'sam3'...
remote: Enumerating objects: 620, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 620 (delta 24), reused 21 (delta 21), pack-reused 588 (from 2)
Receiving objects: 100% (620/620), 58.95 MiB | 23.10 MiB/s, done.
Resolving deltas: 100% (95/95), done.
✅ SAM3 klasörü klonlandı
/content/sam3
Obtaining file:///content/sam3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
... [50 lines truncated for the showcase repo] ...
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninsta


=== Kurulum Kontrolü ===
✅ SAM3 modülleri başarıyla import edildi!


In [5]:
import torch
print(f"GPU var mı? {torch.cuda.is_available()}")
print(f"GPU adı: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Yok'}")
print(f"GPU belleği: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU var mı? True
GPU adı: Tesla T4
GPU belleği: 15.8 GB


In [6]:
# Tüm alt sınıfları içeren hiyerarşik yapı
raw_hierarchy = {
    "Furniture": [
        "sofa", "couch", "armchair", "loveseat", "chair", "dining chair", "office chair",
        "stool", "bench", "ottoman", "pouf", "dining table", "coffee table", "side table",
        "nightstand", "bedside table", "desk", "computer desk", "console table",
        "wardrobe", "closet", "cabinet", "cupboard", "chest of drawers", "dresser",
        "bookshelf", "bookcase", "TV stand", "media unit", "bed frame", "headboard",
        "coat rack", "shoe rack"
    ],
    "Textile": [
        "rug", "carpet", "floor mat", "runner", "doormat", "curtain", "drape", "window blind",
        "pillow", "cushion", "throw pillow", "blanket", "throw blanket", "duvet", "quilt",
        "comforter", "bed spread", "bed sheets", "bedding", "crumpled blanket",
        "towel", "bath mat"
    ],
    "Lighting": [
        "floor lamp", "standing lamp", "reading lamp", "table lamp", "desk lamp",
        "bedside lamp", "ceiling light", "pendant light", "chandelier", "wall sconce",
        "wall lamp", "lampshade"
    ],
    "Decoration": [
        "vase", "flower pot", "planter", "potted plant", "indoor plant", "mirror",
        "wall mirror", "standing mirror", "picture frame", "photo frame", "painting",
        "wall art", "poster", "sculpture", "statue", "figurine", "bust", "candle",
        "candlestick", "candle holder", "decorative bowl", "tray", "basket",
        "storage box", "wall clock", "table clock"
    ],
    "Appliance": [
        "television", "tv screen", "monitor", "display", "refrigerator", "fridge",
        "washing machine", "dryer", "dishwasher", "oven", "stove", "microwave",
        "toaster", "kettle", "coffee maker", "speaker", "soundbar", "air conditioner",
        "fan", "heater", "radiator"
    ]
}

# COCO formatında CATEGORIES listesini oluşturma
CATEGORIES = []
current_id = 1

for supercat, subcats in raw_hierarchy.items():
    for subcat in subcats:
        CATEGORIES.append({
            "id": current_id,
            "name": subcat,
            "supercategory": supercat.lower()
        })
        current_id += 1

# Kontrol: İlk 5 ve toplam sayı
print(f"✅ Toplam Kategori Sayısı: {len(CATEGORIES)}")
print("Örnek (İlk 3):", CATEGORIES[:3])

✅ Toplam Kategori Sayısı: 114
Örnek (İlk 3): [{'id': 1, 'name': 'sofa', 'supercategory': 'furniture'}, {'id': 2, 'name': 'couch', 'supercategory': 'furniture'}, {'id': 3, 'name': 'armchair', 'supercategory': 'furniture'}]


In [7]:
from pathlib import Path

# 1. Ana Yol Tanımlamaları
DRIVE_DATA = Path("/content/drive/MyDrive/furniture-retrieval")

# Test için kullanacağımız klasör
TEST_IMAGES_DIR = DRIVE_DATA / "test_images"

# Çıktıların (JSON ve Maske) kaydedileceği yer (Drive'da ayrı bir klasör açalım karışmasın)
TEST_OUTPUT_DIR = DRIVE_DATA / "test_results"

# 2. Klasörleri Oluştur (Yoksa hata vermez, olanı bozmaz)
for folder in [TEST_IMAGES_DIR, TEST_OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# 3. Test Klasöründeki Resimleri Listele
# Büyük/küçük harf duyarlılığı için farklı uzantıları kontrol eder
image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"]
test_image_files = []

for ext in image_extensions:
    test_image_files.extend(list(TEST_IMAGES_DIR.glob(ext)))

# Alfabetik sırala (Takip etmesi kolay olsun)
test_image_files = sorted(test_image_files)

# 4. Sonuçları Yazdır
print(f"📂 Hedef Klasör: {TEST_IMAGES_DIR}")
print(f"🖼️  Toplam bulunan test resmi: {len(test_image_files)}")

if len(test_image_files) > 0:
    print(f"🔍 Örnek resim: {test_image_files[0].name}")
else:
    print("⚠️ UYARI: test_images klasörü boş veya bulunamadı! Lütfen Drive yolunu kontrol et.")

📂 Hedef Klasör: /content/drive/MyDrive/furniture-retrieval/test_images
🖼️  Toplam bulunan test resmi: 9
🔍 Örnek resim: <image>.jpg


In [ ]:
####################TEST KODU#################
import os
import json
import time
import torch
import numpy as np
from PIL import Image
from datetime import datetime
from pycocotools import mask as mask_utils
from tqdm import tqdm
from pathlib import Path

# --- NMS Fonksiyonu ---
def apply_mask_nms(annotations, iou_threshold=0.5):
    """
    Aynı kategoriye ait maskeler arasında IoU tabanlı NMS uygular.
    """
    if len(annotations) <= 1:
        return annotations

    # Skorlara göre büyükten küçüğe sırala
    sorted_anns = sorted(annotations, key=lambda x: x['score'], reverse=True)

    keep = []
    while sorted_anns:
        best_ann = sorted_anns.pop(0)
        keep.append(best_ann)

        remaining_anns = []
        for ann in sorted_anns:
            # İki RLE maske arasındaki IoU değerini hesapla
            iou = mask_utils.iou([best_ann['segmentation']], [ann['segmentation']], [0])[0][0]

            # Çakışma oranı eşik değerden düşükse (farklı nesne ise) sakla
            if iou < iou_threshold:
                remaining_anns.append(ann)

        sorted_anns = remaining_anns

    return keep

# 1. Model ve Cihaz Ayarları
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Kullanılan cihaz: {device}")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

# 2. Resim Dosyalarını Hazırla
image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"]:
    image_files.extend(list(TEST_IMAGES_DIR.glob(ext)))
image_files = sorted(image_files)

if not image_files:
    raise FileNotFoundError(f"❌ {TEST_IMAGES_DIR} içinde resim bulunamadı!")

# 3. Model Yükleme
print("⏳ SAM 3 Modeli yükleniyor...")
model_load_start = time.time()
model = build_sam3_image_model()
model = model.to(device).eval()
processor = Sam3Processor(model)
print(f"✅ Model hazır! ({time.time() - model_load_start:.2f}s)\n")

# 4. Sayaçlar
total_start_time = time.time()
total_annotations = 0
skipped_count = 0

# 5. Ana İşlem Döngüsü
for img_idx, image_path in enumerate(tqdm(image_files, desc="📸 İşleniyor"), start=1):
    OUTPUT_JSON = TEST_OUTPUT_DIR / (image_path.stem + ".json")

    if OUTPUT_JSON.exists():
        try:
            with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if data.get('annotations'):
                    total_annotations += len(data['annotations'])
                    skipped_count += 1
                    continue
        except: pass

    image_start_time = time.time()
    image_pil = Image.open(image_path).convert("RGB")

    coco_output = {
        "info": {"year": 2024, "version": "1.0", "date_created": datetime.now().strftime("%Y-%m-%d")},
        "images": [{"id": 1, "file_name": image_path.name, "width": image_pil.width, "height": image_pil.height}],
        "annotations": [],
        "categories": CATEGORIES
    }

    image_final_annotations = []

    with torch.no_grad():
        inference_state = processor.set_image(image_pil)

        for category in CATEGORIES:
            output = processor.set_text_prompt(state=inference_state, prompt=category["name"])
            masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

            category_temp_anns = [] # NMS için bu kategoriye özel geçici liste

            if masks is not None and len(masks) > 0:
                for idx, mask in enumerate(masks):
                    score = scores[idx].item() if torch.is_tensor(scores[idx]) else scores[idx]

                    if score < 0.25: continue

                    mask_np = mask.detach().cpu().numpy().squeeze().astype(np.uint8)
                    if mask_np.sum() == 0: continue

                    rle = mask_utils.encode(np.asfortranarray(mask_np))
                    rle["counts"] = rle["counts"].decode("utf-8")

                    if boxes is not None:
                        x1, y1, x2, y2 = boxes[idx].detach().cpu().numpy()
                        bbox = [float(x1), float(y1), float(x2-x1), float(y2-y1)]
                    else:
                        bbox = mask_utils.toBbox(rle).tolist()

                    category_temp_anns.append({
                        "id": 0, # Geçici ID
                        "image_id": 1,
                        "category_id": category["id"],
                        "segmentation": rle,
                        "bbox": bbox,
                        "area": float(mask_utils.area(rle)),
                        "score": float(score),
                        "iscrowd": 0
                    })

            # ⭐ NMS UYGULAMA: Sadece aynı kategori içindeki çakışan maskeleri temizle
            filtered_category_anns = apply_mask_nms(category_temp_anns, iou_threshold=0.5)
            image_final_annotations.extend(filtered_category_anns)

    # 6. ID'leri Düzenle ve Kaydet
    for idx, ann in enumerate(image_final_annotations, 1):
        ann["id"] = idx

    coco_output["annotations"] = image_final_annotations
    image_annotations_count = len(image_final_annotations)

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(coco_output, f, indent=2)

    image_time = time.time() - image_start_time
    total_annotations += image_annotations_count
    tqdm.write(f"✅ [{img_idx}/{len(image_files)}] {image_path.name} | Ann:{image_annotations_count} | Süre:{image_time:.1f}s")

# Final Özeti
total_time = time.time() - total_start_time
print(f"\n{'='*40}\n🎉 İŞLEM TAMAMLANDI!\n{'='*40}")
print(f"📁 Toplam Resim: {len(image_files)}\n⏭️  Atlanan: {skipped_count}\n📊 Toplam Annot: {total_annotations}")
print(f"⏱️  Ortalama Hız: {total_time/max(1, (len(image_files)-skipped_count)):.2f} s/resim")

🚀 Kullanılan cihaz: cuda
⏳ SAM 3 Modeli yükleniyor...
✅ Model hazır! (33.77s)



📸 İşleniyor: 100%|██████████| 9/9 [00:00<00:00, 150.36it/s]


🎉 İŞLEM TAMAMLANDI!
📁 Toplam Resim: 9
⏭️  Atlanan: 9
📊 Toplam Annot: 569
⏱️  Ortalama Hız: 0.07 s/resim


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pycocotools import mask as mask_utils
from pathlib import Path

# 1. Klasör Ayarları
VISUAL_CHECK_DIR = DRIVE_DATA / "test_visual_checks"
VISUAL_CHECK_DIR.mkdir(parents=True, exist_ok=True)

def get_random_color():
    # RGBA formatında: [R, G, B, Alpha] -> Alpha transparanlık için 0.5
    return np.concatenate([np.random.random(3), [0.5]])

# 2. Döngü
json_files = sorted(list(TEST_OUTPUT_DIR.glob("*.json")))
print(f"🖼️  {len(json_files)} resim işleniyor ve kaydediliyor...")

for json_path in json_files:
    with open(json_path, 'r') as f:
        data = json.load(f)

    img_name = data['images'][0]['file_name']
    img_path = TEST_IMAGES_DIR / img_name

    if not img_path.exists(): continue

    # ✅ HATA DÜZELTME: Görüntüyü kesin olarak uint8 (0-255) formatında açıyoruz
    image_pil = Image.open(img_path).convert("RGB")
    image_np = np.array(image_pil).astype(np.uint8)

    fig, ax = plt.subplots(figsize=(12, 8))
    # Ana resmi çiziyoruz (uint8 olduğu için clipping hatası vermez)
    ax.imshow(image_np)

    for ann in data.get('annotations', []):
        mask = mask_utils.decode(ann['segmentation'])
        color = get_random_color()

        # Maske görselleştirme
        # Maskeyi [H, W, 4] boyutunda bir RGBA matrisine çeviriyoruz
        mask_rgba = np.zeros((mask.shape[0], mask.shape[1], 4))
        mask_rgba[mask == 1, :3] = color[:3] # Renk (RGB)
        mask_rgba[mask == 1, 3] = 0.5        # Görünürlük (Alpha)

        # Overlay'i ekle
        ax.imshow(mask_rgba)

        # Bbox ve Etiket
        bbox = ann['bbox']
        cat_id = ann['category_id']
        cat_name = next((c['name'] for c in CATEGORIES if c['id'] == cat_id), "N/A")

        rect = plt.Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3],
                             fill=False, edgecolor=color[:3], linewidth=2)
        ax.add_patch(rect)
        ax.text(bbox[0], bbox[1] - 5, f"{cat_name}", color='white',
                fontsize=9, fontweight='bold',
                bbox=dict(facecolor=color[:3], alpha=0.8, pad=2))

    plt.axis('off')

    # 💾 DOSYAYI KAYDET
    save_filename = f"check_{img_path.stem}.png"
    save_path = VISUAL_CHECK_DIR / save_filename
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=120)

    # Ekranda da bir tanesini gösterip temizleyelim
    # plt.show() # Hepsini ekrana basmak istemiyorsan bunu yorum satırı yapabilirsin
    plt.close(fig) # RAM dolmaması için her resimden sonra figürü kapatıyoruz

print(f"✅ İşlem bitti. Görselleri şuradan kontrol edebilirsin: {VISUAL_CHECK_DIR}")

🖼️  9 resim işleniyor ve kaydediliyor...
✅ İşlem bitti. Görselleri şuradan kontrol edebilirsin: /content/drive/MyDrive/furniture-retrieval/test_visual_checks


In [9]:
import os
import json
import time
import torch
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime
from pycocotools import mask as mask_utils
from tqdm import tqdm
from pathlib import Path

# --- 1. AYARLAR VE YOLLAR ---
DRIVE_DATA = Path("/content/drive/MyDrive/furniture-retrieval")
PARQUET_PATH = DRIVE_DATA / "golden_dataset/golden_dataset.parquet"
LOCAL_DATA_DIR = DRIVE_DATA / "dataset" # Ziplerin açıldığı yerel klasör
OUTPUT_DIR = DRIVE_DATA / "golden_annotations" # Sonuçların kaydedileceği yer

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 2. NMS FONKSİYONU ---
def apply_mask_nms(annotations, iou_threshold=0.5):
    if len(annotations) <= 1:
        return annotations
    # Skorlara göre sırala
    sorted_anns = sorted(annotations, key=lambda x: x['score'], reverse=True)
    keep = []
    while sorted_anns:
        best_ann = sorted_anns.pop(0)
        keep.append(best_ann)
        remaining_anns = []
        for ann in sorted_anns:
            iou = mask_utils.iou([best_ann['segmentation']], [ann['segmentation']], [0])[0][0]
            if iou < iou_threshold:
                remaining_anns.append(ann)
        sorted_anns = remaining_anns
    return keep

# --- 3. MODELİ YÜKLE ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Cihaz: {device}")

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("⏳ SAM 3 Modeli yükleniyor...")
model = build_sam3_image_model().to(device).eval()
processor = Sam3Processor(model)
print("✅ Model ve İşlemci hazır.")

# --- 4. DATASETİ YÜKLE ---
df_golden = pd.read_parquet(PARQUET_PATH)
print(f"📊 Golden Dataset yüklendi: {len(df_golden)} adet seçilmiş resim işlenecek.")

# --- 5. ANA ANNOTATION DÖNGÜSÜ ---
total_start_time = time.time()

for idx, row in tqdm(df_golden.iterrows(), total=len(df_golden), desc="🔥 Processing Golden Dataset"):
    # Parquet'teki folder_name ve image_name'i kullanarak resmi yerel diskte bul
    image_path = LOCAL_DATA_DIR / row['folder_name'] / row['image_name']

    # Çıktı adı: klasör_resim.json (Çakışmaları önlemek için)
    output_filename = f"{row['folder_name']}_{Path(row['image_name']).stem}.json"
    output_json_path = OUTPUT_DIR / output_filename

    # Eğer daha önce işlendiyse atla (Checkpoint sistemi)
    if output_json_path.exists():
        continue

    if not image_path.exists():
        # print(f"⚠️ Dosya bulunamadı: {image_path}")
        continue

    try:
        image_pil = Image.open(image_path).convert("RGB")

        # COCO Formatı Başlangıcı + Parquet'ten gelen meta veriler
        coco_output = {
            "image_meta": {
                "folder": row['folder_name'],
                "filename": row['image_name'],
                "selection_method": row['selection_method'],
                "original_embedding": row['embedding'].tolist() if hasattr(row['embedding'], 'tolist') else row['embedding']
            },
            "annotations": [],
            "categories": CATEGORIES # Önceki adımda tanımladığımız 114 sınıf
        }

        image_final_annotations = []

        with torch.no_grad():
            # ⭐ OPTİMİZASYON: Görüntüyü bir kez işle
            inference_state = processor.set_image(image_pil)

            # 114 Kategorinin her biri için tara
            for category in CATEGORIES:
                output = processor.set_text_prompt(state=inference_state, prompt=category["name"])
                masks, scores = output["masks"], output["scores"]

                category_temp_anns = []
                if masks is not None and len(masks) > 0:
                    for i, mask in enumerate(masks):
                        score = scores[i].item() if torch.is_tensor(scores[i]) else scores[i]

                        if score < 0.30: continue # Güven eşiği

                        mask_np = mask.detach().cpu().numpy().squeeze().astype(np.uint8)
                        if mask_np.sum() == 0: continue

                        rle = mask_utils.encode(np.asfortranarray(mask_np))
                        rle["counts"] = rle["counts"].decode("utf-8")

                        category_temp_anns.append({
                            "category_id": category["id"],
                            "segmentation": rle,
                            "bbox": mask_utils.toBbox(rle).tolist(),
                            "score": float(score),
                            "area": float(mask_utils.area(rle))
                        })

                # ⭐ NMS: Kategori içindeki çakışmaları temizle
                filtered_anns = apply_mask_nms(category_temp_anns, iou_threshold=0.5)
                image_final_annotations.extend(filtered_anns)

        # ID'leri düzenle ve kaydet
        for i, ann in enumerate(image_final_annotations, 1):
            ann["id"] = i

        coco_output["annotations"] = image_final_annotations

        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(coco_output, f, indent=2)

    except Exception as e:
        print(f"❌ Hata ({image_path.name}): {e}")

total_time = time.time() - total_start_time
print(f"\n✅ İŞLEM TAMAMLANDI!")
print(f"⏱️ Toplam Süre: {total_time/3600:.2f} saat")
print(f"📂 Sonuçlar şuraya kaydedildi: {OUTPUT_DIR}")

🚀 Cihaz: cuda
⏳ SAM 3 Modeli yükleniyor...
✅ Model ve İşlemci hazır.
📊 Golden Dataset yüklendi: 40000 adet seçilmiş resim işlenecek.


🔥 Processing Golden Dataset: 100%|██████████| 40000/40000 [00:35<00:00, 1136.71it/s]


✅ İŞLEM TAMAMLANDI!
⏱️ Toplam Süre: 0.01 saat
📂 Sonuçlar şuraya kaydedildi: /content/drive/MyDrive/furniture-retrieval/golden_annotations
